# Les files — Exercice 5 : les VIN d'abord

**Notions travaillées :** répartir les éléments d'une file selon un critère, file auxiliaire.

Les **VIN** (*Very Important Numbers*) sont les multiples de 12. On veut les placer **en premier** dans la file. On choisit de **conserver l'ordre** d'origine à l'intérieur de chaque groupe (c'est ce qu'on attend d'une file d'attente : un VIN ne double que les non-VIN).

Exemple : `5 24 7 12 36 1` → `24 12 36 5 7 1`.

## Mise en place

On importe la classe `File` écrite à l'exercice 1 (le fichier `file.py` doit être **dans le même dossier** que ce notebook).

L'énoncé interdit d'utiliser le type `list` : on ne manipule les files **qu'avec leurs méthodes** `enfile`, `defile`, `taille`, `est_vide`. La petite fonction `file_de` sert uniquement à fabriquer rapidement des files pour les tests.

On affiche une file sous la forme `tête < ... < queue` : on **défile à gauche** et on **enfile à droite**.

In [1]:
from file import File   # la classe File de l'exercice 1 (fichier file.py placé à côté du notebook)


def file_de(*elements):
    """Outil pour les tests : construit une file contenant les éléments donnés (le premier en tête)."""
    f = File()
    for el in elements:
        f.enfile(el)
    return f


print(file_de("a", "b", "c"))

tête < 'a' < 'b' < 'c' < queue


## Reconnaître un VIN

`n` est un multiple de 12 lorsque le reste de la division de `n` par 12 est nul : `n % 12 == 0`.

💡 0 est un multiple de 12 (0 = 12 × 0) : c'est donc un VIN. Les multiples négatifs (−24…) aussi.

In [2]:
def est_vin(n):
    return n % 12 == 0


assert est_vin(24) and est_vin(0) and est_vin(-36)
assert not est_vin(5) and not est_vin(6)

## La fonction `vin_first(f)`

**Idée :** comme à l'aéroport, on fait **deux files** : les VIN restent dans `f`, les autres patientent dans une file auxiliaire `autres`. Puis on remet les `autres` **derrière** les VIN.

**Algorithme**, avec un tour complet de `f` :
1. pour chaque élément défilé : si c'est un VIN, on le **ré-enfile dans `f`** ; sinon, on l'enfile dans `autres` ;
2. après le tour, `f` ne contient plus que les VIN, dans l'ordre ;
3. on vide `autres` dans `f` : les non-VIN se placent derrière.

| Étape | `f` | `autres` |
|:---|:---|:---|
| départ | `5 24 7 12 36 1` | |
| après le tour | `24 12 36` | `5 7 1` |
| après le vidage de `autres` | `24 12 36 5 7 1` | |

💡 Ici, on peut vider `autres` avec `while not autres.est_vide()`, car on **ne ré-enfile pas** dans `autres` : la boucle s'arrête bien.

In [3]:
def vin_first(f):
    autres = File()
    for _ in range(f.taille()):
        el = f.defile()
        if est_vin(el):
            f.enfile(el)          # un VIN reste dans f
        else:
            autres.enfile(el)     # les autres patientent
    while not autres.est_vide():
        f.enfile(autres.defile())


f = file_de(5, 24, 7, 12, 36, 1)
vin_first(f)
print(f)
assert repr(f) == repr(file_de(24, 12, 36, 5, 7, 1))

for g, attendu in [(file_de(1, 2, 3), file_de(1, 2, 3)),        # aucun VIN
                   (file_de(12, 24), file_de(12, 24)),          # que des VIN
                   (file_de(7, 0), file_de(0, 7)),              # 0 est un VIN
                   (File(), File())]:
    vin_first(g)
    assert repr(g) == repr(attendu)
print("Tous les tests passent ✔")

tête < 24 < 12 < 36 < 5 < 7 < 1 < queue
Tous les tests passent ✔


⚠️ **Erreur fréquente :** mettre les VIN dans une file auxiliaire **et** les autres dans une autre file, puis oublier de tout remettre dans `f`. L'énoncé demande de **modifier** la file reçue.